# Análisis EDA + Modelo Canónico — Fase 2
**Proyecto:** bank-pipeline · **Objetivo:** producir TODOS los números y figuras del apartado e del informe.

Reutiliza `model_api` (features + pipeline) para que análisis == producción.

> Ejecutar con **Restart & Run All** para reproducibilidad completa.

## Tarea 1 — Celda 0: Imports, rutas y reuso de `model_api`

In [ ]:
import json, os, sys
from pathlib import Path

# Anclar al root del repositorio para que model_api sea importable
_here = Path(os.getcwd())
if _here.name == 'ml':
    os.chdir(_here.parent)
REPO = Path(os.getcwd())
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — evita bloqueo de plt.show()
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, roc_curve,
)

from model_api.features import (
    NUMERIC_FEATURES, NOMINAL_FEATURES, BOOLEAN_FEATURES,
    FEATURE_COLUMNS, TARGET, select_features,
)
from model_api.training import build_pipeline, _gini

RANDOM_STATE = 42
REPORTS = Path('reports')
FIGS = REPORTS / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

# Columnas para EDA (duration incluida aquí, NO en el modelo)
EDA_NUMERIC = NUMERIC_FEATURES + ['duration']
EDA_CATEGORICAL = NOMINAL_FEATURES + BOOLEAN_FEATURES

print('FEATURE_COLUMNS:', FEATURE_COLUMNS)
print(f'{len(FEATURE_COLUMNS)} features (sin duration)')

## Tarea 2 — Celda 1: Carga de datos (CSV o DB)

In [ ]:
def cargar_bank_clean():
    csv_path = Path('ml/data/bank_clean_export.csv')
    if csv_path.exists():
        df = pd.read_csv(csv_path)
    else:
        from model_api import db
        df = db.fetch_retrain_source()

    # Convertir 'True'/'False' strings a booleanos para deposit y bool features
    bool_cols = BOOLEAN_FEATURES + [TARGET]
    for col in bool_cols:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].map({'True': True, 'False': False})
    return df

df = cargar_bank_clean()
print('Filas:', len(df))
print('Columnas:', df.columns.tolist())
print('Dtypes:\n', df.dtypes)

## Tarea 3 — Celda 2: Análisis de calidad de datos (apartado e.calidad)

In [ ]:
print('=== Shape ===')
print(df.shape)

print('\n=== Nulos por columna ===')
print(df.isna().sum())

print('\n=== Estadísticas descriptivas (numéricas incl. duration) ===')
desc = df[EDA_NUMERIC].describe(percentiles=[0.25, 0.5, 0.75])
display(desc)

print('\n=== Balance del TARGET ===')
balance = df[TARGET].value_counts(dropna=False)
balance_pct = df[TARGET].value_counts(normalize=True).mul(100).round(1)
print(balance)
print()
print(balance_pct)

# Guardamos para eda_summary
_n_si = int((df[TARGET] == True).sum())
_n_no = int((df[TARGET] == False).sum())
_pct_si = round(_n_si / len(df) * 100, 1)
print(f'\nBalance: {_pct_si}% sí / {100-_pct_si}% no')

print('\nNota: no se imputa en modelado porque el ETL ya hizo dropna en columnas')
print('críticas; unknown se mantiene como categoría válida.')

## Tarea 4 — Celda 3: Análisis univariado (apartado e.univariado)

In [ ]:
# Histogramas de variables numéricas
for col in ['age', 'balance', 'duration']:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='white')
    ax.set_title(f'Distribución de {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')
    plt.tight_layout()
    plt.savefig(FIGS / f'hist_{col}.png', dpi=120)
    plt.show()
    plt.close()
    print(f'Guardado: hist_{col}.png')

# Barras de frecuencia categóricas
for col in ['job', 'education']:
    freq = df[col].value_counts()
    fig, ax = plt.subplots(figsize=(9, 4))
    freq.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Frecuencia de {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('N')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(FIGS / f'bar_{col}.png', dpi=120)
    plt.show()
    plt.close()
    print(f'Guardado: bar_{col}.png')

# Balance del target
target_counts = df[TARGET].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
target_counts.plot(kind='bar', ax=ax, color=['#d9534f', '#5cb85c'], edgecolor='white')
ax.set_title('Balance del target: deposit')
ax.set_xlabel('deposit')
ax.set_ylabel('N')
ax.set_xticklabels([str(v) for v in target_counts.index], rotation=0)
for i, v in enumerate(target_counts):
    ax.text(i, v + 50, f'{v}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(FIGS / 'target_balance.png', dpi=120)
plt.show()
plt.close()
print('Guardado: target_balance.png')

## Tarea 5 — Celda 4: Bivariado numérico + matriz de correlación (apartado e.bivariado)

In [ ]:
print('=== Numéricas por grupo de deposit ===')
_target_int = df[TARGET].astype(int)
_num_df = df[EDA_NUMERIC].copy()
_num_df['deposit'] = _target_int
display(_num_df.groupby('deposit')[EDA_NUMERIC].agg(['mean', 'median', 'std']).round(2))

# Matriz de correlación (numéricas + deposit como 0/1)
num = df[EDA_NUMERIC].copy()
num['deposit'] = _target_int
corr = num.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Matriz de correlación (numéricas + deposit)')
plt.tight_layout()
plt.savefig(FIGS / 'correlacion.png', dpi=120)
plt.show()
plt.close()
print('Guardado: correlacion.png')

_dur_corr = corr.loc['duration', 'deposit']
print(f'\nCorrelacion duration<->deposit: {_dur_corr:.3f}')
print('NOTA: duration es la variable de mayor asociacion con el target, pero se')
print('      EXCLUYE del modelo por fuga (data leakage): solo se conoce post-llamada.')

## Tarea 6 — Celda 5: Bivariado categórico

In [ ]:
_df_plot = df.copy()
_df_plot['deposit_int'] = df[TARGET].astype(int)

for col in EDA_CATEGORICAL:
    ct = pd.crosstab(_df_plot[col], _df_plot[TARGET], normalize='index') * 100
    if True in ct.columns:
        tasa = ct[True].sort_values(ascending=False)
    else:
        tasa = ct[ct.columns[-1]].sort_values(ascending=False)
    print(f'\n--- {col} --- tasa de suscripción (%)')
    print(tasa.round(1))

# Figuras para las más informativas
for col in ['job', 'education', 'poutcome', 'month']:
    ct = pd.crosstab(_df_plot[col], _df_plot[TARGET], normalize='index') * 100
    if True in ct.columns:
        tasa = ct[True].sort_values(ascending=False)
    else:
        tasa = ct[ct.columns[-1]].sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9, 4))
    tasa.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Tasa de suscripción por {col} (%)')
    ax.set_xlabel(col)
    ax.set_ylabel('% suscripción')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(FIGS / f'biv_{col}.png', dpi=120)
    plt.show()
    plt.close()
    print(f'Guardado: biv_{col}.png')

## Tarea 7 — Celda 6: Partición + preprocesamiento (apartado e.partición/preproc)

In [ ]:
data = df.dropna(subset=[TARGET]).copy()
y = data[TARGET].astype(int)
X = select_features(data)   # sin duration — paridad con producción

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print('X no contiene duration:', 'duration' not in X.columns)
print('Columnas de X:', X.columns.tolist())
print(f'Train: {len(X_tr)} · Test: {len(X_te)}')
print(f'Balance train  y=1: {y_tr.mean():.3f} · y=0: {1-y_tr.mean():.3f}')
print(f'Balance test   y=1: {y_te.mean():.3f} · y=0: {1-y_te.mean():.3f}')

# Preprocesador que espeja training.build_pipeline (Opción A — sin tocar model_api)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SKPipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

_numeric_pipe = SKPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
_nominal_pipe = SKPipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
_boolean_pipe = SKPipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
])
preprocessor = ColumnTransformer([
    ('num', _numeric_pipe, NUMERIC_FEATURES),
    ('nom', _nominal_pipe, NOMINAL_FEATURES),
    ('bool', _boolean_pipe, BOOLEAN_FEATURES),
])
print('\nPreprocesador construido (espejo exacto de training.build_pipeline)')

## Tarea 8 — Celda 7: Comparación de algoritmos (apartado e.modelo)

In [ ]:
from sklearn.pipeline import Pipeline as SKPipeline

def _eval_pipeline(pipe, name):
    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        'model': name,
        'f1':      round(float(f1_score(y_te, pred, zero_division=0)), 4),
        'roc_auc': round(float(roc_auc_score(y_te, proba)), 4),
        'precision': round(float(precision_score(y_te, pred, zero_division=0)), 4),
        'recall': round(float(recall_score(y_te, pred, zero_division=0)), 4),
    }

# Baseline: Regresión Logística
import copy
_pre_lr = copy.deepcopy(preprocessor)
_lr = SKPipeline([
    ('pre', _pre_lr),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced',
                               random_state=RANDOM_STATE)),
])
res_lr = _eval_pipeline(_lr, 'LogisticRegression')

# Candidato fuerte: RandomForest de producción
res_rf = _eval_pipeline(build_pipeline(), 'RandomForest')

# Tabla comparativa
_comparison = [res_lr, res_rf]

# XGBoost opcional
try:
    from xgboost import XGBClassifier
    import copy
    _pre_xgb = copy.deepcopy(preprocessor)
    _xgb_pipe = SKPipeline([
        ('pre', _pre_xgb),
        ('clf', XGBClassifier(n_estimators=200, random_state=RANDOM_STATE,
                              eval_metric='logloss', verbosity=0)),
    ])
    res_xgb = _eval_pipeline(_xgb_pipe, 'XGBoost')
    _comparison.append(res_xgb)
except ImportError:
    print('XGBoost no instalado — comparación solo LR vs RF')

df_comp = pd.DataFrame(_comparison).set_index('model')
print('\n=== Tabla comparativa de modelos ===')
display(df_comp)

print()
print('Justificación de elección — RandomForest:')
print('  • Mejor F1 y AUC que Regresión Logística (captura relaciones no lineales).')
print('  • Mejor recall en clase positiva: reduce falsos negativos (clientes')
print('    que SÍ suscribirán pero el modelo clasifica como no).')
print('  • Robusto a variables ruidosas y sin supuesto de linealidad.')
print('  • Alineado al objetivo de negocio: maximizar contactos con prospecto real.')

## Tarea 9 — Celda 8: Modelo titular + métricas (apartado e.métricas)

In [ ]:
pipe = build_pipeline()   # RF de producción, sin duration
pipe.fit(X_tr, y_tr)

proba = pipe.predict_proba(X_te)[:, 1]
pred = (proba >= 0.5).astype(int)
cm = confusion_matrix(y_te, pred)   # [[tn, fp], [fn, tp]]
auc = roc_auc_score(y_te, proba)

metrics = {
    'accuracy':  round(float(accuracy_score(y_te, pred)), 4),
    'precision': round(float(precision_score(y_te, pred, zero_division=0)), 4),
    'recall':    round(float(recall_score(y_te, pred, zero_division=0)), 4),
    'f1':        round(float(f1_score(y_te, pred, zero_division=0)), 4),
    'roc_auc':   round(float(auc), 4),
    'gini':      round(float(_gini(auc)), 4),
}

tn, fp, fn, tp = cm.ravel()

print('=== Métricas del modelo titular (RandomForest sin duration) ===')
for k, v in metrics.items():
    print(f'  {k:<12}: {v}')

print(f'\nMatriz de confusión:')
print(f'  TN={tn}  FP={fp}')
print(f'  FN={fn}  TP={tp}')

print('\nNota: AUC realista (sin duration). ~0.89–0.95 requeriría duration (fuga).')

## Tarea 10 — Celda 9: Figuras de métricas (ROC, Confusión, Importancia)

In [ ]:
# Curva ROC
fpr, tpr, _ = roc_curve(y_te, proba)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2,
        label=f'RandomForest (AUC={metrics["roc_auc"]:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos')
ax.set_title('Curva ROC — Modelo titular (sin duration)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(FIGS / 'roc.png', dpi=120)
plt.show()
plt.close()
print('Guardado: roc.png')

# Matriz de confusión
fig, ax = plt.subplots(figsize=(5, 4))
_cm_df = pd.DataFrame(
    cm, index=['Real No', 'Real Sí'], columns=['Pred No', 'Pred Sí']
)
sns.heatmap(_cm_df, annot=True, fmt='d', cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('Matriz de confusión — RandomForest')
plt.tight_layout()
plt.savefig(FIGS / 'confusion.png', dpi=120)
plt.show()
plt.close()
print('Guardado: confusion.png')

# Importancia de variables
_pre_fitted = pipe.named_steps['pre']
_clf = pipe.named_steps['clf']
_feat_names = _pre_fitted.get_feature_names_out()
_importances = _clf.feature_importances_
_imp_ser = pd.Series(_importances, index=_feat_names).sort_values(ascending=False)[:20]
fig, ax = plt.subplots(figsize=(9, 6))
_imp_ser[::-1].plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 20 variables más importantes — RandomForest')
ax.set_xlabel('Importancia media (impurity)')
plt.tight_layout()
plt.savefig(FIGS / 'importancia.png', dpi=120)
plt.show()
plt.close()
print('Guardado: importancia.png')

## Tarea 11 — Celda 10: Exportar artefactos para informe y dashboard

In [ ]:
# model_metrics.json
model_metrics = {
    'dataset': {
        'source': 'bank_clean',
        'n_rows': len(df),
        'target': TARGET,
        'balance': {
            'si':     _n_si,
            'no':     _n_no,
            'si_pct': _pct_si,
        },
    },
    'split': {
        'test_size':    0.2,
        'stratified':   True,
        'random_state': RANDOM_STATE,
        'n_train':      int(len(X_tr)),
        'n_test':       int(len(X_te)),
    },
    'model': {
        'name':             'RandomForestClassifier',
        'duration_excluded': True,
        'class_weight':     'balanced',
        'n_estimators':     200,
    },
    'metrics': metrics,
    'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
    'model_comparison': [
        {'model': r['model'], 'f1': r['f1'], 'roc_auc': r['roc_auc']}
        for r in _comparison
    ],
}

_json_path = REPORTS / 'model_metrics.json'
with open(_json_path, 'w', encoding='utf-8') as f:
    json.dump(model_metrics, f, indent=2, ensure_ascii=False)
print(f'Guardado: {_json_path}')

# eda_summary.md
_dur_corr_val = corr.loc['duration', 'deposit']
_biv_table = _num_df.groupby('deposit')[EDA_NUMERIC].agg(['mean', 'median']).round(2)

_summary_lines = [
    '# EDA Summary — bank-pipeline Fase 2\n',
    '\n',
    '## 1. Dataset\n',
    f'- **Filas:** {len(df)}  \n',
    f'- **Target:** `{TARGET}`  \n',
    f'- **Balance:** {_pct_si}% sí / {100-_pct_si}% no '
    '— dataset balanceado (~50/50), no aplicar SMOTE.  \n',
    '\n',
    '## 2. Calidad de datos\n',
    '- Sin valores nulos en columnas críticas (ETL ya hizo `dropna`).  \n',
    '- `unknown` se conserva como categoría válida.  \n',
    '\n',
    '## 3. Estadísticas descriptivas (numéricas)\n',
]
_summary_lines.append(desc.to_markdown() + '\n\n')

_summary_lines += [
    '## 4. Bivariado numérico por deposit\n',
]
_summary_lines.append(_biv_table.to_markdown() + '\n\n')

_summary_lines += [
    '## 5. Variable `duration` (nota de fuga)\n',
    f'- Correlación `duration` ↔ `deposit`: **{_dur_corr_val:.3f}** '
    '(la más alta del dataset).  \n',
    '- **Excluida del modelo** por fuga de datos: solo se conoce post-llamada.  \n',
    '- Aparece en EDA únicamente como referencia.  \n',
    '\n',
    '## 6. Modelo titular\n',
    '- **Algoritmo:** RandomForestClassifier (200 árboles, class_weight=balanced).  \n',
    '- **Features:** 15 columnas (sin `duration`).  \n',
    f'- **Accuracy:**  {metrics["accuracy"]}  \n',
    f'- **Precision:** {metrics["precision"]}  \n',
    f'- **Recall:**    {metrics["recall"]}  \n',
    f'- **F1:**        {metrics["f1"]}  \n',
    f'- **ROC-AUC:**   {metrics["roc_auc"]}  \n',
    f'- **Gini:**      {metrics["gini"]}  \n',
    '\n',
    '## 7. Limitaciones conocidas\n',
    '- Dataset balanceado (~50/50): accuracy es informativo; '
    'class_weight=balanced tiene efecto mínimo.  \n',
    '- 11k filas: entrenado sobre el total (sin muestreo).  \n',
    '- Reproducible: random_state=42 en todo.  \n',
]

_md_path = REPORTS / 'eda_summary.md'
with open(_md_path, 'w', encoding='utf-8') as f:
    f.writelines(_summary_lines)
print(f'Guardado: {_md_path}')

print('\n=== RESUMEN FINAL ===')
print(f'Filas dataset:  {len(df)}')
print(f'Balance target: {_pct_si}% sí / {100-_pct_si}% no')
print(f'Train/Test:     {len(X_tr)}/{len(X_te)}')
print(f'ROC-AUC:        {metrics["roc_auc"]} | F1: {metrics["f1"]} | Gini: {metrics["gini"]}')
print('Artefactos generados:')
for p in sorted(FIGS.glob('*.png')):
    print(f'  {p}')
print(f'  {_json_path}')
print(f'  {_md_path}')